### 1. Install Necessary Libraries
We need `face-recognition` for the core logic and `opencv-python` for image handling. Note: `face-recognition` depends on `dlib`, which can take a few minutes to compile if not cached.

In [ ]:
!pip install face-recognition opencv-python

### 2. Setup known faces and Attendance Logic
This script will load known images, encode them, and provide a function to mark attendance in a CSV file.

In [ ]:
import face_recognition
import cv2
import numpy as np
import os
from datetime import datetime
import pandas as pd

# Path to the directory containing images of people you want to recognize
# For this demo, we assume you have a folder 'known_faces' or will upload images.
# Let's create a placeholder structure.
KNOWN_FACES_DIR = 'known_faces'
if not os.path.exists(KNOWN_FACES_DIR):
    os.makedirs(KNOWN_FACES_DIR)

def mark_attendance(name):
    with open('attendance.csv', 'a+') as f:
        f.seek(0)
        datalist = f.readlines()
        namelist = [line.split(',')[0] for line in datalist]
        if name not in namelist:
            now = datetime.now()
            dt_string = now.strftime('%H:%M:%S')
            f.writelines(f'\n{name},{dt_string}')
            print(f'Marked attendance for {name}')

print('Setup complete. Please upload images of known people to the /known_faces folder.')

### 3. Encode Known Faces
This cell will iterate through your `known_faces` folder, learn how to recognize them, and store their encodings.

In [ ]:
known_face_encodings = []
known_face_names = []

def load_known_faces():
    for filename in os.listdir(KNOWN_FACES_DIR):
        if filename.endswith(('.jpg', '.png', '.jpeg')):
            img = face_recognition.load_image_file(f'{KNOWN_FACES_DIR}/{filename}')
            encoding = face_recognition.face_encodings(img)[0]
            known_face_encodings.append(encoding)
            known_face_names.append(os.path.splitext(filename)[0])
    print(f'Encoded {len(known_face_names)} faces: {known_face_names}')

# Call this after you have uploaded images to the folder
# load_known_faces()

### 4. Recognition and Attendance Loop
This function takes an image (from a webcam or file), finds faces, and marks attendance if a match is found.

In [ ]:
def process_attendance_frame(frame):
    if not known_face_encodings:
        return "No known faces loaded. Please run load_known_faces() first."

    # Resize for faster processing
    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

    face_locations = face_recognition.face_locations(rgb_small_frame)
    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)

    result_name = "Unknown"
    for face_encoding, face_location in zip(face_encodings, face_locations):
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
        name = "Unknown"

        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_face_names[best_match_index]
                mark_attendance(name)

        # Store the captured face image
        save_captured_face(frame, face_location, name)
        result_name = name

    return result_name

### 4b. Store Captured Faces
This cell ensures a directory exists to save images of faces detected during the attendance process.

In [ ]:
import os
import cv2
from datetime import datetime

CAPTURED_FACES_DIR = 'captured_faces'
if not os.path.exists(CAPTURED_FACES_DIR):
    os.makedirs(CAPTURED_FACES_DIR)

def save_captured_face(frame, face_location, name):
    # Scale back up by 4 since we process a 1/4 size image for detection
    top, right, bottom, left = [v * 4 for v in face_location]
    face_image = frame[top:bottom, left:right]
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{CAPTURED_FACES_DIR}/{name}_{timestamp}.jpg"
    cv2.imwrite(filename, face_image)
    return filename

I will now update the recognition loop to use this saving function.

### 5. Test with Webcam (Colab Specific)
Run this cell to capture a photo from your webcam. It will then pass the image to our `process_attendance_frame` function.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      try {
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        await new Promise((resolve) => capture.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', quality);
      } catch (err) {
        return 'ERROR:' + err.name + ': ' + err.message;
      }
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))

  if data.startswith('ERROR:'):
    raise Exception(data)

  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  test_image = cv2.imread(filename)
  result_name = process_attendance_frame(test_image)
  print(f'Recognition Result: {result_name}')

except Exception as err:
  print(f'Webcam Error: {err}. Please ensure browser camera permissions are enabled.')

### 6. View Attendance Log
Run this cell to see the list of people who have been marked present.

In [ ]:
import pandas as pd

def view_attendance():
    if os.path.exists('attendance.csv'):
        df = pd.read_csv('attendance.csv', names=['Name', 'Time'])
        return df
    else:
        return "No attendance recorded yet."

view_attendance()